# Modeling Continuous-Time Linear Systems with `scipy.signal`

This notebook demonstrates how to model continuous-time linear time-invariant (LTI) systems using `scipy.signal`, focusing on a **1st-order RC Low-Pass Filter**.

### Table of Contents
1. **Mathematical Derivation & S-Domain Representation**
2. **Continuous-Time System Definition (`signal.TransferFunction`)**
3. **Bode Plot Generation (`signal.bode`)**
4. **Time-Domain Simulation with Discrete Data Samples (`signal.lsim`)**

## 1. Mathematical Derivation

For a 1st-order series RC low-pass filter, the voltage transfer function in the frequency domain is derived via the voltage divider relationship:

$$
H(j\omega) = \frac{V_{\text{out}}(j\omega)}{V_{\text{in}}(j\omega)} = \frac{\frac{1}{j\omega C}}{R + \frac{1}{j\omega C}} = \frac{1}{1 + j\omega RC}
$$

By mapping $j\omega \to s$ (Laplace domain complex frequency variable $s = \sigma + j\omega$ along the imaginary axis where $\sigma = 0$):

$$
H(s) = \frac{1}{1 + sRC} = \frac{\frac{1}{RC}}{s + \frac{1}{RC}} = \frac{\omega_c}{s + \omega_c}
$$

where $\tau = RC$ is the circuit time constant, $\omega_c = 1/\tau$ is the cutoff angular frequency (rad/s), and $f_c = \frac{\omega_c}{2\pi}$ is the cutoff frequency in Hertz.

In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

# Set plot style
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 2. Define Continuous-Time System in `scipy.signal`

In `scipy.signal`, continuous-time transfer functions are represented by 1D arrays or lists of polynomial coefficients in descending powers of $s$:
- **Numerator (`num`)**: $[b_0] = [1/\tau]$
- **Denominator (`den`)**: $[a_1, a_0] = [1, 1/\tau]$ (representing $1\cdot s^1 + \frac{1}{\tau}\cdot s^0$)

In [ ]:
# Circuit parameter specifications
R = 1.0e3   # 1 kOhm (1000 Ohms)
C = 1.0e-6  # 1 uF (1e-6 Farads)

# Time constant & Cutoff frequency
tau = R * C               # Time constant tau = 1.0 ms
wc = 1.0 / tau            # Cutoff angular frequency = 1000 rad/s
fc = wc / (2.0 * np.pi)   # Cutoff frequency ~ 159.15 Hz

print(f"Time constant (tau) : {tau * 1e3:.2f} ms")
print(f"Cutoff frequency (fc): {fc:.2f} Hz ({wc:.1f} rad/s)")

# Transfer function polynomials: H(s) = (1/tau) / (s + 1/tau)
num = [1.0 / tau]
den = [1.0, 1.0 / tau]

# Instantiate continuous-time LTI system object
rc_filter = signal.TransferFunction(num, den)
print("\nSystem Transfer Function:")
print(rc_filter)

## 3. Frequency Response (Bode Plot)

We use `signal.bode()` to compute the angular frequency vector $\omega$ (in rad/s), logarithmic magnitude response (in dB), and phase response (in degrees).

In [ ]:
# Compute Bode frequency response
w, mag, phase = signal.bode(rc_filter, n=500)
freq_hz = w / (2.0 * np.pi)  # Convert rad/s to Hz

fig, (ax_mag, ax_phase) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, layout='constrained')

# 1. Magnitude Plot
ax_mag.semilogx(freq_hz, mag, color='navy', lw=2)
ax_mag.axvline(fc, color='crimson', linestyle='--', label=f'Cutoff $f_c$ ({fc:.1f} Hz)')
ax_mag.axhline(-3.01, color='gray', linestyle=':', label='-3 dB Level')
ax_mag.set_ylabel('Magnitude [dB]')
ax_mag.set_title('Bode Diagram: 1st Order RC Low-Pass Filter')
ax_mag.grid(True, which='both', alpha=0.4)
ax_mag.legend(loc='lower left')

# 2. Phase Plot
ax_phase.semilogx(freq_hz, phase, color='darkorange', lw=2)
ax_phase.axvline(fc, color='crimson', linestyle='--')
ax_phase.axhline(-45.0, color='gray', linestyle=':', label='-45° at Cutoff')
ax_phase.set_xlabel('Frequency [Hz]')
ax_phase.set_ylabel('Phase [degrees]')
ax_phase.set_yticks([0, -30, -45, -60, -90])
ax_phase.grid(True, which='both', alpha=0.4)
ax_phase.legend(loc='lower left')

plt.show()

## 4. Applying Filter to Discrete Sampled Data (`signal.lsim`)

We construct a composite discrete time series with three sinusoidal components:
1. **$f_1 = 20\text{ Hz}$** (Passband): expected gain $\approx 0\text{ dB}$ ($1.0$), phase lag $\approx -7^\circ$
2. **$f_2 = 160\text{ Hz}$** (Near cutoff): expected gain $\approx -3\text{ dB}$ ($0.707$), phase lag $\approx -45^\circ$
3. **$f_3 = 1500\text{ Hz}$** (Stopband): expected gain $\approx -19.5\text{ dB}$ ($< 0.1$), phase lag $\approx -84^\circ$

`signal.lsim()` simulates the continuous-time linear system's response to discrete input samples using linear interpolation.

In [ ]:
# Time vector definition
fs = 20000.0         # 20 kHz sampling frequency
duration = 0.05      # 50 ms duration
t = np.linspace(0.0, duration, int(fs * duration), endpoint=False)

# Construct composite input signal
f1, f2, f3 = 20.0, 160.0, 1500.0
x1 = 1.0 * np.sin(2.0 * np.pi * f1 * t)
x2 = 1.0 * np.sin(2.0 * np.pi * f2 * t)
x3 = 1.0 * np.sin(2.0 * np.pi * f3 * t)
x_in = x1 + x2 + x3

# Continuous-time simulation on sampled input
t_out, y_out, _ = signal.lsim(rc_filter, U=x_in, T=t)

# Plot input vs filtered response
fig, (ax_in, ax_out) = plt.subplots(2, 1, figsize=(10, 6), sharex=True, layout='constrained')

# Input Signal
ax_in.plot(t * 1e3, x_in, color='black', lw=1.2, label='Input $x(t) = \sin(2\pi f_1 t) + \sin(2\pi f_2 t) + \sin(2\pi f_3 t)$')
ax_in.set_ylabel('Amplitude [V]')
ax_in.set_title('Input Waveform with 20 Hz, 160 Hz, and 1500 Hz Tones')
ax_in.grid(True, alpha=0.3)
ax_in.legend(loc='upper right')

# Filtered Output
ax_out.plot(t_out * 1e3, y_out, color='teal', lw=1.6, label='Filtered Output $y(t)$')
ax_out.plot(t * 1e3, x1, color='crimson', linestyle=':', lw=1.2, alpha=0.7, label='Pure 20 Hz Reference')
ax_out.set_xlabel('Time [ms]')
ax_out.set_ylabel('Amplitude [V]')
ax_out.set_title('Filtered Output Waveform (via signal.lsim)')
ax_out.grid(True, alpha=0.3)
ax_out.legend(loc='upper right')

plt.show()